# Corrected pipeline smoke test
## SMOKE TEST — NOT A REPORTED RESULT
This notebook is disabled by default. When explicitly enabled on a GPU Kaggle session, it validates all six canonical files and hashes, then uses only 96 Bangla training rows, 48 validation rows, and one epoch. It does not read the test split for evaluation. Proxy auxiliary labels are explicitly enabled only to verify head/loss wiring; they are not gold supervision.

In [ ]:
from pathlib import Path
import json, subprocess, sys

RUN_HEAVY = False
CONFIG_NAME = 'corrected_smoke_mdistilbert_multitask.json'

def find_repo():
    candidates = [Path('/kaggle/working/Capstone-Project')]
    candidates.extend(Path('/kaggle/input').glob('*/Capstone-Project'))
    candidates.extend(path for path in Path('/kaggle/input').glob('*') if (path / 'corrected_pipeline').is_dir())
    for candidate in candidates:
        if (candidate / 'corrected_pipeline' / 'runner.py').is_file():
            return candidate
    raise FileNotFoundError('Attach the complete Capstone-Project repository to this Kaggle notebook.')

if not RUN_HEAVY:
    subprocess.run([sys.executable, '-m', 'compileall', '-q', 'corrected_pipeline', 'tests_stage1a'], cwd=repo, check=True)
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests_stage1a', '-v'], cwd=repo, check=True)
    print('SMOKE TEST — NOT A REPORTED RESULT')
    print('Guard active: no model, tokenizer, checkpoint, or dataset was loaded.')
else:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a Kaggle GPU before running the smoke test.')
    repo = find_repo()
    config_path = repo / 'configs' / CONFIG_NAME
    config = json.loads(config_path.read_text(encoding='utf-8'))
    assert config['run_kind'] == 'smoke'
    assert config['result_status'] == 'SMOKE TEST — NOT A REPORTED RESULT'
    assert config['training']['epochs'] == 1
    assert config['execution']['evaluate_test'] is False
    missing = [path for path in config['dataset']['paths'].values() if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Missing canonical Kaggle files: ' + ', '.join(missing))
    print('SMOKE TEST — NOT A REPORTED RESULT')
    subprocess.run([sys.executable, '-m', 'corrected_pipeline.runner', '--config', str(config_path)], cwd=repo, check=True)